# 01 - Exploratory data analysis
Shape, class balance, duplicates, missing values, message length, word frequency, examples.

**Note:** if `data/raw/SMSSpamCollection` is absent, only the small hand-written demo dataset is analysed (see docs/dataset_documentation.md).

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
import matplotlib; matplotlib.rcParams["figure.figsize"] = (6, 3.5)
import matplotlib.pyplot as plt, pandas as pd, numpy as np

In [ ]:
from src.data.load_data import load_training_data
raw = load_training_data(use_uci=True)   # uses data/raw/SMSSpamCollection if you placed it there, plus the demo set
print("sources:", raw.source.value_counts().to_dict(), "| provenance:", raw.provenance.value_counts().to_dict())

In [ ]:
print("shape:", raw.shape)
print("missing values:\n", raw[["message","label"]].isna().sum())
print("duplicate messages:", raw.message.duplicated().sum())

In [ ]:
from src.data.preprocess import clean_dataframe
df, removed = clean_dataframe(raw)
print("removed during cleaning:", removed, "-> rows:", len(df))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df.label.value_counts().plot.bar(ax=ax[0], title="Original labels")
df.target.map({0: "legitimate", 1: "flagged"}).value_counts().plot.bar(ax=ax[1], title="Binary target (imbalance)")
plt.tight_layout()

In [ ]:
df["chars"] = df.message.str.len()
for t, name in [(0, "legitimate"), (1, "flagged")]:
    plt.hist(df[df.target == t].chars, bins=30, alpha=0.6, label=name)
plt.legend(); plt.title("Message length (characters)")

In [ ]:
from collections import Counter
from src.nlp.text_processing import tokenize
for t, name in [(0, "legitimate"), (1, "flagged")]:
    c = Counter(w for m in df[df.target == t].message for w in tokenize(m))
    print(name, c.most_common(15))

In [ ]:
from src.nlp.language_detection import detect_language
df["language"] = df.message.map(lambda m: detect_language(m).code)
print(pd.crosstab(df.language, df.target.map({0: "legitimate", 1: "flagged"})))

In [ ]:
for lab, g in df.groupby("label"):
    print("\n==", lab)
    for m in g.message.head(3): print("  -", m[:140])